# Avance 02 · C5 · Etapa 1: delimitación
**Equipo A&C — Inteligencia de Negocios**

En este notebook delimitamos los incidentes viales C5 a **fecha de creación desde 2018-01-01** y a las **16 alcaldías de la CDMX**. Conservamos los archivos originales y generamos evidencia de todas las exclusiones. No deduplicamos ni mezclamos los incidentes C5 con los hechos SSC: sus unidades y criterios de registro deben analizarse por separado.

## Inventario real
El ZIP `CARPETA_DATOS.zip` contiene `inViales_2014_2015.csv`, `inViales_2016_2018.csv`, `inViales_2019_2021.csv` e `inViales_2022_2024.csv`. Cada uno tiene 17 columnas. **No confiamos en el periodo del nombre**: hay fechas de creación de otros años dentro de algunos archivos.

## 1. Localización de datos y configuración
Podemos ejecutar este notebook desde otra carpeta de JupyterLab: si no encontramos el ZIP, podemos señalar la carpeta `C5` extraída. Los resultados se escriben en `resultados/resultados_C5/etapa1`.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import re, unicodedata, zipfile
from IPython.display import display

RUTA_ZIP_MANUAL = None  # Ejemplo: Path('/home/usuario/CARPETA_DATOS.zip')
RUTA_C5_MANUAL = None   # Alternativa: Path('/home/usuario/CARPETA_DATOS/C5')
ARCHIVOS = ['inViales_2014_2015.csv','inViales_2016_2018.csv','inViales_2019_2021.csv','inViales_2022_2024.csv']
subruta = 'CARPETA_DATOS/C5/'

if RUTA_ZIP_MANUAL:
    zip_path = Path(RUTA_ZIP_MANUAL).expanduser().resolve()
else:
    raiz = Path.cwd().resolve()
    candidatos = [p/'CARPETA_DATOS.zip' for p in [raiz,*raiz.parents]]
    candidatos += list(Path.home().glob('**/CARPETA_DATOS.zip'))
    zip_path = next((p for p in candidatos if p.is_file()), None)

carpeta_c5 = Path(RUTA_C5_MANUAL).expanduser().resolve() if RUTA_C5_MANUAL else None
if zip_path is None and carpeta_c5 is None:
    candidatos = list(Path.home().glob('**/C5/inViales_2019_2021.csv'))
    carpeta_c5 = candidatos[0].parent if len(candidatos) == 1 else None
if zip_path is None and carpeta_c5 is None:
    raise FileNotFoundError('Sube CARPETA_DATOS.zip o configura RUTA_ZIP_MANUAL / RUTA_C5_MANUAL.')
if zip_path is not None:
    with zipfile.ZipFile(zip_path) as z:
        faltantes = [f for f in ARCHIVOS if subruta+f not in z.namelist()]
    assert not faltantes, f'Faltan archivos en ZIP: {faltantes}'
else:
    assert all((carpeta_c5/f).is_file() for f in ARCHIVOS)

# El directorio de salida queda junto al ZIP o en el directorio de trabajo.
raiz_proyecto = zip_path.parent if zip_path else Path.cwd()
SALIDA = raiz_proyecto/'resultados'/'resultados_C5'/'etapa1'
SALIDA.mkdir(parents=True,exist_ok=True)
print('Origen:', zip_path if zip_path else carpeta_c5)
print('Evidencias:', SALIDA)

Origen: /workspaces/Proyecto-sobre-An-lisis-de-hechos-de-transito-de-la-CDMX-/Avance02/CARPETA_DATOS.zip
Evidencias: /workspaces/Proyecto-sobre-An-lisis-de-hechos-de-transito-de-la-CDMX-/Avance02/resultados/resultados_C5/etapa1


## 2. Reglas de alcaldía y fechas
Usamos preferentemente `alcaldia_catalogo` porque es el campo catalogado de ubicación. Si está vacío, probamos `alcaldia_inicio` y después `alcaldia_cierre`; **no sustituimos una alcaldía por otra cuando hay conflicto**: dejamos constancia en un indicador de discrepancia. Si ninguna es válida, excluimos la fila del subconjunto delimitado y la conservamos en evidencia.

Las fechas inválidas, anteriores a 2018 y posteriores a la fecha actual se separan para revisión. No excluimos fechas futuras por la fecha del sistema sin antes registrarlas explícitamente.

In [4]:
ALCALDIAS = {'ALVARO OBREGON','AZCAPOTZALCO','BENITO JUAREZ','COYOACAN','CUAJIMALPA DE MORELOS','CUAUHTEMOC','GUSTAVO A MADERO','IZTACALCO','IZTAPALAPA','LA MAGDALENA CONTRERAS','MIGUEL HIDALGO','MILPA ALTA','TLAHUAC','TLALPAN','VENUSTIANO CARRANZA','XOCHIMILCO'}
EQUIVALENCIAS = {'CUAJIMALPA':'CUAJIMALPA DE MORELOS','MAGDALENA CONTRERAS':'LA MAGDALENA CONTRERAS','GUSTAVO A. MADERO':'GUSTAVO A MADERO','GUSTAVO A MADERO':'GUSTAVO A MADERO','ALVARO OBREGON':'ALVARO OBREGON'}
def normalizar(valor):
    if pd.isna(valor): return pd.NA
    s = unicodedata.normalize('NFKD',str(valor)).encode('ascii','ignore').decode('ascii').upper().strip()
    s = re.sub(r'[^A-Z0-9 ]',' ',s)
    s = re.sub(r'\s+',' ',s).strip()
    s = EQUIVALENCIAS.get(s,s)
    return s if s in ALCALDIAS else pd.NA

def leer_csv(nombre):
    if zip_path:
        with zipfile.ZipFile(zip_path) as z:
            with z.open(subruta+nombre) as f:
                return pd.read_csv(f,dtype=str,encoding='utf-8-sig',low_memory=False)
    return pd.read_csv(carpeta_c5/nombre,dtype=str,encoding='utf-8-sig',low_memory=False)

muestras=[]
for nombre in ARCHIVOS:
    prueba = leer_csv(nombre)
    muestras.append({'archivo':nombre,'filas':len(prueba),'columnas':len(prueba.columns),'columnas_reales':' | '.join(prueba.columns)})
    del prueba
inventario = pd.DataFrame(muestras)
display(inventario)
inventario.to_csv(SALIDA/'inventario_crudos_C5.csv',index=False,encoding='utf-8-sig')

,archivo,filas,columnas,columnas_reales
0,inViales_2014_2015.csv,356072,17,folio | fecha_creacion | hora_creacion | dia_s...
1,inViales_2016_2018.csv,664111,17,folio | fecha_creacion | hora_creacion | dia_s...
2,inViales_2019_2021.csv,590636,17,folio | fecha_creacion | hora_creacion | dia_s...
3,inViales_2022_2024.csv,504261,17,folio | fecha_creacion | hora_creacion | dia_s...


## 3. Delimitación y evidencia por archivo
Procesamos los archivos por separado, identificamos por qué se excluye cada fila y conservamos sus valores originales. Añadimos columnas de procedencia, fecha y alcaldía normalizada **solo en las salidas**. Para fechas utilizamos `%Y-%m-%d`, compatible con versiones de pandas anteriores a 2.0.

In [5]:
resumen=[]
limpios=[]
excluidos=[]
columnas_base=None
for nombre in ARCHIVOS:
    df=leer_csv(nombre)
    if columnas_base is None: columnas_base=list(df.columns)
    if list(df.columns)!=columnas_base:
        raise ValueError(f'Esquema distinto en {nombre}; revisar antes de concatenar')
    df['archivo_origen']=nombre
    df['fila_origen']=np.arange(2,len(df)+2)
    fecha=pd.to_datetime(df['fecha_creacion'].str.strip(),format='%Y-%m-%d',errors='coerce')
    df['fecha_creacion_validada']=fecha.dt.strftime('%Y-%m-%d')
    for col in ['alcaldia_catalogo','alcaldia_inicio','alcaldia_cierre']:
        df[col+'_norm']=df[col].map(normalizar)
    df['alcaldia_normalizada']=df['alcaldia_catalogo_norm'].fillna(df['alcaldia_inicio_norm']).fillna(df['alcaldia_cierre_norm'])
    valores=df[['alcaldia_catalogo_norm','alcaldia_inicio_norm','alcaldia_cierre_norm']]
    df['alcaldias_discrepantes']=valores.apply(lambda x: len(set(x.dropna()))>1,axis=1)
    f_invalida=fecha.isna()
    f_anterior=fecha.notna() & (fecha<pd.Timestamp('2018-01-01'))
    alcaldia_invalida=df['alcaldia_normalizada'].isna()
    df['motivo_exclusion']=''
    df.loc[f_invalida,'motivo_exclusion']+='fecha_invalida;'
    df.loc[f_anterior,'motivo_exclusion']+='fecha_anterior_2018;'
    df.loc[alcaldia_invalida,'motivo_exclusion']+='sin_alcaldia_valida;'
    aptas=df['motivo_exclusion'].eq('')
    buenos=df.loc[aptas].copy()
    malos=df.loc[~aptas].copy()
    limpios.append(buenos)
    excluidos.append(malos)
    resumen.append({'archivo':nombre,'filas_recibidas':len(df),'filas_aptas':len(buenos),'filas_excluidas':len(malos),'fechas_invalidas':int(f_invalida.sum()),'anteriores_2018':int(f_anterior.sum()),'sin_alcaldia_valida':int(alcaldia_invalida.sum()),'alcaldias_discrepantes':int(df['alcaldias_discrepantes'].sum()),'fechas_posteriores_hoy_alerta':int((fecha>pd.Timestamp.today().normalize()).sum())})
    assert len(df)==len(buenos)+len(malos)
    del df

resumen_df=pd.DataFrame(resumen)
display(resumen_df)
print('Totales:')
display(resumen_df.select_dtypes(include='number').sum().to_frame('total').T)
resumen_df.to_csv(SALIDA/'resumen_delimitacion_por_archivo_C5.csv',index=False,encoding='utf-8-sig')

,archivo,filas_recibidas,filas_aptas,filas_excluidas,fechas_invalidas,anteriores_2018,sin_alcaldia_valida,alcaldias_discrepantes,fechas_posteriores_hoy_alerta
0,inViales_2014_2015.csv,356072,0,356072,0,356072,41,21649,0
1,inViales_2016_2018.csv,664111,246561,417550,0,417547,10,38020,0
2,inViales_2019_2021.csv,590636,590621,15,0,0,15,31508,0
3,inViales_2022_2024.csv,504261,504237,24,0,0,24,29673,0


Totales:


,filas_recibidas,filas_aptas,filas_excluidas,fechas_invalidas,anteriores_2018,sin_alcaldia_valida,alcaldias_discrepantes,fechas_posteriores_hoy_alerta
total,2115080,1341419,773661,0,773619,90,120850,0


## 4. Exportación y comprobaciones
Unimos únicamente las filas aptas, manteniendo el nombre del archivo y el número de fila original. Los registros excluidos se exportan aparte. **No quitamos folios repetidos ni duplicados exactos** en esta etapa.

In [6]:
datos_delimitados=pd.concat(limpios,ignore_index=True)
datos_excluidos=pd.concat(excluidos,ignore_index=True)
assert len(datos_delimitados)+len(datos_excluidos)==int(resumen_df.filas_recibidas.sum())
assert datos_delimitados['alcaldia_normalizada'].isin(ALCALDIAS).all()
fechas_ok=pd.to_datetime(datos_delimitados['fecha_creacion_validada'],format='%Y-%m-%d',errors='coerce')
assert fechas_ok.notna().all() and fechas_ok.ge(pd.Timestamp('2018-01-01')).all()
assert len(datos_delimitados)==int(resumen_df.filas_aptas.sum())

ruta_aptos=SALIDA/'incidentes_viales_C5_etapa1.csv'
ruta_excluidos=SALIDA/'incidentes_viales_C5_excluidos_etapa1.csv'
datos_delimitados.to_csv(ruta_aptos,index=False,encoding='utf-8-sig')
datos_excluidos.to_csv(ruta_excluidos,index=False,encoding='utf-8-sig')
print('Guardados:',ruta_aptos,ruta_excluidos,sep='\n')
print('Filas aptas:',len(datos_delimitados),'| excluidas:',len(datos_excluidos))
print('Rango de fechas:',fechas_ok.min().date(),fechas_ok.max().date())
print('Años:',fechas_ok.dt.year.value_counts().sort_index().to_dict())

Guardados:
/workspaces/Proyecto-sobre-An-lisis-de-hechos-de-transito-de-la-CDMX-/Avance02/resultados/resultados_C5/etapa1/incidentes_viales_C5_etapa1.csv
/workspaces/Proyecto-sobre-An-lisis-de-hechos-de-transito-de-la-CDMX-/Avance02/resultados/resultados_C5/etapa1/incidentes_viales_C5_excluidos_etapa1.csv
Filas aptas: 1341419 | excluidas: 773661
Rango de fechas: 2018-01-01 2024-02-29
Años: {2018: 246577, 2019: 229494, 2020: 162671, 2021: 198513, 2022: 234914, 2023: 233325, 2024: 35925}


## 5. Perfil de calidad inicial
Elaboramos frecuencias de alcaldía, año, motivo de exclusión y tipo de incidente para orientar la Etapa 2. Las categorías de incidentes **no son equivalentes automáticamente** a lesiones o fallecimientos confirmados.

In [7]:
datos_delimitados['alcaldia_normalizada'].value_counts(dropna=False).rename_axis('alcaldia').reset_index(name='filas').to_csv(SALIDA/'frecuencias_alcaldias_C5.csv',index=False,encoding='utf-8-sig')
fechas_ok.dt.year.value_counts().sort_index().rename_axis('anio').reset_index(name='filas').to_csv(SALIDA/'frecuencias_anios_C5.csv',index=False,encoding='utf-8-sig')
datos_delimitados['incidente_c4'].value_counts(dropna=False).rename_axis('incidente_c4').reset_index(name='filas').to_csv(SALIDA/'frecuencias_incidentes_C5.csv',index=False,encoding='utf-8-sig')
datos_excluidos['motivo_exclusion'].value_counts(dropna=False).rename_axis('motivo').reset_index(name='filas').to_csv(SALIDA/'motivos_exclusion_C5.csv',index=False,encoding='utf-8-sig')
print('Distribución por año:')
display(fechas_ok.dt.year.value_counts().sort_index().to_frame('filas'))
print('Alcaldías discrepantes entre columnas (conservadas para revisión):',int(datos_delimitados.alcaldias_discrepantes.sum()))
print('Archivos de evidencia:',len(list(SALIDA.glob('*.csv'))))

Distribución por año:


,filas
fecha_creacion_validada,
2018,246577
2019,229494
2020,162671
2021,198513
2022,234914
2023,233325
2024,35925


Alcaldías discrepantes entre columnas (conservadas para revisión): 76204
Archivos de evidencia: 8


In [8]:
print("COMPROBACIÓN DE LA ETAPA 1 DE C5")

print("\nRegistros conservados:")
print(len(datos_delimitados))

print("\nRegistros excluidos:")
print(len(datos_excluidos))

print("\nPrimeros registros:")
display(datos_delimitados.head())

print("\nResumen:")
display(resumen_df)

COMPROBACIÓN DE LA ETAPA 1 DE C5

Registros conservados:
1341419

Registros excluidos:
773661

Primeros registros:


,folio,fecha_creacion,hora_creacion,dia_semana,fecha_cierre,hora_cierre,tipo_incidente_c4,incidente_c4,alcaldia_inicio,codigo_cierre,...,latitud,archivo_origen,fila_origen,fecha_creacion_validada,alcaldia_catalogo_norm,alcaldia_inicio_norm,alcaldia_cierre_norm,alcaldia_normalizada,alcaldias_discrepantes,motivo_exclusion
0,C5/180101/02118,2018-01-01,06:47:06,Lunes,2018-01-01,06:53:53,Accidente,Motociclista,COYOACAN,D,...,19.30278996,inViales_2016_2018.csv,417536,2018-01-01,TLALPAN,COYOACAN,COYOACAN,TLALPAN,True,
1,C5/180101/07447,2018-01-01,21:16:23,Lunes,2018-01-01,23:32:50,Accidente,Choque sin lesionados,AZCAPOTZALCO,A,...,19.49078997,inViales_2016_2018.csv,417537,2018-01-01,AZCAPOTZALCO,AZCAPOTZALCO,AZCAPOTZALCO,AZCAPOTZALCO,False,
2,C5/180101/03894,2018-01-01,12:43:21,Lunes,2018-01-01,13:48:52,Accidente,Choque sin lesionados,MIGUEL HIDALGO,A,...,19.45972998,inViales_2016_2018.csv,417538,2018-01-01,MIGUEL HIDALGO,MIGUEL HIDALGO,MIGUEL HIDALGO,MIGUEL HIDALGO,False,
3,C5/180101/03234,2018-01-01,10:48:29,Lunes,2018-01-01,11:15:41,Accidente,Choque sin lesionados,IZTAPALAPA,N,...,19.33549002,inViales_2016_2018.csv,417539,2018-01-01,IZTAPALAPA,IZTAPALAPA,IZTAPALAPA,IZTAPALAPA,False,
4,C5/180101/00022,2018-01-01,00:02:31,Lunes,2018-01-01,00:11:24,Lesionado,Atropellado,IZTAPALAPA,N,...,19.30689999,inViales_2016_2018.csv,417540,2018-01-01,IZTAPALAPA,IZTAPALAPA,IZTAPALAPA,IZTAPALAPA,False,



Resumen:


,archivo,filas_recibidas,filas_aptas,filas_excluidas,fechas_invalidas,anteriores_2018,sin_alcaldia_valida,alcaldias_discrepantes,fechas_posteriores_hoy_alerta
0,inViales_2014_2015.csv,356072,0,356072,0,356072,41,21649,0
1,inViales_2016_2018.csv,664111,246561,417550,0,417547,10,38020,0
2,inViales_2019_2021.csv,590636,590621,15,0,0,15,31508,0
3,inViales_2022_2024.csv,504261,504237,24,0,0,24,29673,0


## 6. Registro de decisiones y conclusión
**Decisiones aplicadas:** delimitación temporal por `fecha_creacion`; 16 alcaldías normalizadas; preferencia por `alcaldia_catalogo` con respaldo de inicio/cierre cuando falta; exclusiones registradas; originales intactos; duplicados no eliminados.

**Pendientes para Etapa 2:** comprobar integridad de folios, registros exactos repetidos, discrepancias entre alcaldías, campos vacíos, horarios, coordenadas y posibles superposiciones entre periodos. Verificar que el evento C5 y el hecho SSC no se contabilicen como unidades intercambiables.

En esta primera etapa **delimitamos** los incidentes viales de C5 para nuestro proyecto y generamos evidencias reproducibles. No afirmamos todavía que la información esté libre de duplicados o de problemas de calidad.